# master_data.csv 생성
## 지정학적 위기 시 비트코인은 디지털 금인가?
### 캡스톤디자인 | 팀명: 분석많이된다

---

## 결측치 원인 및 처리 전략

| 결측 원인 | 건수 | 처리 방법 |
|---|---|---|
| 주말 (토·일) | 91건 | **제거** — NYSE 거래일 아님 |
| 미국 공휴일 | 9건 | **제거** — NYSE 거래일 아님 |
| hormuz_crisis 기간 (returns 시작일 문제) | 32건 | **Step 1에서 재수집** (2019-05-01~) |
| VIX 간헐적 누락 | 소수 | **ffill** |
| Fear & Greed 간헐적 누락 | 소수 | **ffill** |

## 출력 파일

```
master_data.csv
  컬럼: date | BTC | Gold | TLT | DXY | SP500 | NASDAQ
         GPR_custom | GPR | GPR_zscore
         VIX | fear_greed | fear_greed_lag1
         event_name | event_date
```

## 흐름

```
Step 0. 라이브러리
Step 1. returns.csv 재수집 (2019-05-01~, hormuz_crisis 커버)
Step 2. gpr_combined.csv 로드
Step 3. VIX 수집
Step 4. Fear & Greed 수집
Step 5. 병합
Step 6. 결측치 처리
Step 7. 검증 및 저장
```

---
## Step 0. 라이브러리 설치 및 로드

In [9]:
# 최초 1회 실행
# !pip install yfinance pandas numpy requests --quiet

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
import os
warnings.filterwarnings('ignore')

# 분석 대상 이벤트
EVENT_DATES = {
    'hormuz_crisis'          : '2019-06-13',
    'soleimani_assassination': '2020-01-03',
    'russia_ukraine_invasion': '2022-02-24',
    'israel_hamas'           : '2023-10-07',
    'israel_iran'            : '2024-04-01',
    'us_israel_iran'         : '2026-02-28',
}

# 수집 기간
# hormuz_crisis 이벤트 기간 2019-05-18~ 커버를 위해 2019-05-01부터
GLOBAL_START = '2019-01-01'
GLOBAL_END   = '2026-04-28'

print('✅ 라이브러리 로드 완료')
print(f'   수집 기간: {GLOBAL_START} ~ {GLOBAL_END}')

✅ 라이브러리 로드 완료
   수집 기간: 2019-01-01 ~ 2026-04-28


---
## Step 1. returns.csv 재수집

**기존 문제:** returns.csv가 2019-07-02부터 시작 → hormuz_crisis 기간(2019-05-18~) 대부분 결측

**해결:** 2019-05-01부터 재수집하여 returns.csv 덮어쓰기

In [3]:
TICKERS = {
    'BTC'   : 'BTC-USD',
    'Gold'  : 'GC=F',
    'TLT'   : 'TLT',
    'DXY'   : 'DX-Y.NYB',
    'SP500' : '^GSPC',
    'NASDAQ': '^IXIC',
}

print('▶ yfinance 가격 데이터 수집 중...')
raw = yf.download(
    list(TICKERS.values()),
    start=GLOBAL_START,
    end=GLOBAL_END,
    auto_adjust=False,
    progress=False
)['Close']
# 컬럼 순서 매칭 안전하게 수행
raw = raw[list(TICKERS.values())]
raw.columns = list(TICKERS.keys())

# 1. 거래일(SP500 기준)과 비거래일 분리
is_trading_day = raw['SP500'].notna()
trading_days = raw[is_trading_day].index
non_trading_days = raw[~is_trading_day].index

# 2. 비트코인(BTC) 전처리: 비거래일 수익을 직전 거래일에 합산하기 위한 작업
# BTC 가격의 로그 수익률을 먼저 구함 (모든 날짜 포함)
btc_all_returns = np.log(raw['BTC'] / raw['BTC'].shift(1))

# 3. [수정된 로직] 비거래일의 수익을 '직전 거래일'로 몰아서 평균 처리
# 로직: 금요일에 진입하면 토, 일요일을 확인하여 (금+토+일)/3 처리 후 주말은 건너뜀
processed_btc_returns = []
all_dates = raw.index.tolist()

i = 0
while i < len(all_dates):
    current_date = all_dates[i]
    
    # 현재 날짜가 거래일(SP500 데이터가 있는 날)인 경우
    if is_trading_day.loc[current_date]:
        combined_return = btc_all_returns.loc[current_date]
        if pd.isna(combined_return): combined_return = 0
        
        # 다음 날부터 비거래일이 얼마나 이어지는지 확인
        count = 1
        j = i + 1
        while j < len(all_dates) and not is_trading_day.loc[all_dates[j]]:
            next_ret = btc_all_returns.loc[all_dates[j]]
            if not pd.isna(next_ret):
                combined_return += next_ret
            count += 1
            j += 1
        
        # (현재 거래일 수익 + 이후 붙어있는 모든 비거래일 수익) / 전체 일수
        avg_return = combined_return / count
        processed_btc_returns.append(avg_return)
        
        # 비거래일만큼 인덱스를 건너뜀
        i = j
    else:
        # 데이터의 첫 시작이 비거래일인 경우 예외 처리
        i += 1

# 4. 최종 데이터프레임 구성 (주식 시장 거래일 기준)
# 비트코인 외 자산들은 원래대로 거래일 데이터만 추출
prices = raw.loc[trading_days].ffill()
returns = np.log(prices / prices.shift(1))

# 비트코인 열을 위에서 계산된 '선반영 평균 수익률'로 교체
# 첫날(NaN)을 제외한 길이를 맞추기 위해 조정
if len(processed_btc_returns) > len(returns):
    # returns는 첫날이 dropna로 날아가므로, processed_btc_returns도 첫 번째 거래일 결과 제외
    returns['BTC'] = processed_btc_returns[1:]
else:
    returns['BTC'] = processed_btc_returns

returns = returns.dropna()
returns.index.name = 'date'

# 저장
returns.to_csv('returns.csv', encoding='utf-8-sig')

print(f'✅ returns.csv 업데이트 완료')
print(f'   기간: {returns.index.min().date()} ~ {returns.index.max().date()}')
print(f'   행 수: {len(returns)}일')
print(f'   결측치: {returns.isnull().sum().sum()}건')
print(f'\n기술통계:')
print(returns.describe().round(4))

▶ yfinance 가격 데이터 수집 중...
✅ returns.csv 업데이트 완료
   기간: 2019-01-03 ~ 2026-04-27
   행 수: 1838일
   결측치: 0건

기술통계:
             BTC       Gold        TLT        DXY      SP500     NASDAQ
count  1838.0000  1838.0000  1838.0000  1838.0000  1838.0000  1838.0000
mean      0.0010     0.0007    -0.0002     0.0000     0.0006     0.0007
std       0.0335     0.0113     0.0102     0.0043     0.0125     0.0151
min      -0.4647    -0.1207    -0.0690    -0.0214    -0.1277    -0.1315
25%      -0.0129    -0.0043    -0.0064    -0.0025    -0.0044    -0.0060
50%       0.0002     0.0010     0.0000     0.0000     0.0010     0.0014
75%       0.0155     0.0065     0.0058     0.0025     0.0067     0.0087
max       0.1718     0.0591     0.0725     0.0164     0.0909     0.1148


---
## Step 2. gpr_combined.csv 로드

GPR_custom_analysis.ipynb에서 생성한 파일 (315행, 6개 이벤트 × 52~55일)

In [5]:
GPR_FILE = 'gpr_combined.csv'

if not os.path.exists(GPR_FILE):
    raise FileNotFoundError(
        f'{GPR_FILE} 없음.\nGPR_custom_analysis.ipynb를 먼저 실행하세요.'
    )

gpr = pd.read_csv(GPR_FILE)
gpr['date'] = pd.to_datetime(gpr['date'])

for col in ['F3_z','F3_raw','GPR','GPR_zscore','N','mean_tone']:
    if col in gpr.columns:
        gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

# F3_z → GPR_custom 명시적 생성
gpr['GPR_custom'] = gpr['F3_z']

print('✅ gpr_combined.csv 로드 완료')
print(f'   행 수: {len(gpr)}')
print(f'   기간: {gpr["date"].min().date()} ~ {gpr["date"].max().date()}')
print(f'\n이벤트별 일수:')
print(gpr.groupby('event_name')['date'].count().to_string())

✅ gpr_combined.csv 로드 완료
   행 수: 315
   기간: 2019-05-18 ~ 2026-03-26

이벤트별 일수:
event_name
hormuz_crisis              52
israel_hamas               52
israel_iran                52
russia_ukraine_invasion    52
soleimani_assassination    52
us_israel_iran             55


---
## Step 3. VIX 수집

GARCH Model 3·4의 외생변수로 사용돼요.

In [4]:
print('▶ VIX 수집 중...')
vix_raw = yf.download(
    '^VIX',
    start=GLOBAL_START,
    end=GLOBAL_END,
    
    auto_adjust=False,
    progress=False
)['Close']

if hasattr(vix_raw, 'squeeze'):
    vix_raw = vix_raw.squeeze()

vix = pd.DataFrame({'VIX': vix_raw})
vix.index = pd.to_datetime(vix.index)
vix.index.name = 'date'
vix['VIX'] = pd.to_numeric(vix['VIX'], errors='coerce')
vix = vix.dropna().reset_index()

print(f'✅ VIX 수집 완료')
print(f'   기간: {vix["date"].min().date()} ~ {vix["date"].max().date()}')
print(f'   행 수: {len(vix)}')
print(vix['VIX'].describe().round(2))

▶ VIX 수집 중...
✅ VIX 수집 완료
   기간: 2019-01-02 ~ 2026-04-27
   행 수: 1839
count    1839.00
mean       20.21
std         7.52
min        11.54
25%        15.35
50%        18.31
75%        23.06
max        82.69
Name: VIX, dtype: float64


---
## Step 4. Fear & Greed Index 수집

- 범위: 0(극도의 공포) ~ 100(극도의 탐욕)
- GARCH 외생변수로 `fear_greed_lag1` (전일값) 사용

In [6]:
def fetch_fear_greed(limit=3000):
    url = f'https://api.alternative.me/fng/?limit={limit}&format=json'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()['data']
        fg = pd.DataFrame(data)
        fg['date']       = pd.to_datetime(fg['timestamp'].astype(int), unit='s').dt.normalize()
        fg['fear_greed'] = fg['value'].astype(int)
        fg['fg_label']   = fg['value_classification']
        fg = (fg[['date','fear_greed','fg_label']]
              .sort_values('date')
              .drop_duplicates(subset=['date'])
              .reset_index(drop=True))
        return fg
    except Exception as e:
        print(f'⚠️ API 오류: {e}')
        return pd.DataFrame(columns=['date','fear_greed','fg_label'])

print('▶ Fear & Greed 수집 중...')
fg = fetch_fear_greed(limit=3000)

if len(fg) > 0:
    fg['date'] = pd.to_datetime(fg['date'])
    fg_range = fg[fg['date'] >= pd.Timestamp(GLOBAL_START)]
    print(f'✅ Fear & Greed 수집 완료')
    print(f'   전체: {fg["date"].min().date()} ~ {fg["date"].max().date()}')
    print(f'   분석 기간 내: {len(fg_range)}일')
    print(fg_range['fear_greed'].describe().round(1))
else:
    print('⚠️ 수집 실패 → NaN으로 처리됩니다')

▶ Fear & Greed 수집 중...
✅ Fear & Greed 수집 완료
   전체: 2018-02-08 ~ 2026-04-30
   분석 기간 내: 2676일
count    2676.0
mean       47.9
std        22.2
min         5.0
25%        28.0
50%        49.0
75%        68.0
max        95.0
Name: fear_greed, dtype: float64


---
## Step 5. 병합

GPR 날짜(이벤트 기간)를 기준으로 LEFT JOIN해요.

```
gpr_combined (315행) ← 기준
  LEFT JOIN returns
  LEFT JOIN vix
  LEFT JOIN fear_greed
```

In [7]:
# ── 날짜 컬럼 통일 ───────────────────────────────────
returns_df = returns.copy().reset_index()
if 'Date' in returns_df.columns:
    returns_df = returns_df.rename(columns={'Date': 'date'})
returns_df['date'] = pd.to_datetime(returns_df['date'])

vix_df = vix.copy()
vix_df['date'] = pd.to_datetime(vix_df['date'])

fg_df = fg.copy() if len(fg) > 0 else pd.DataFrame(columns=['date','fear_greed','fg_label'])
fg_df['date'] = pd.to_datetime(fg_df['date'])

# ── GPR 사용 컬럼 선택 ───────────────────────────────
gpr_cols = ['date','event_name','event_date',
            'GPR_custom','F3_raw','GPR','GPR_zscore','N','mean_tone']
gpr_use = gpr[[c for c in gpr_cols if c in gpr.columns]].copy()

# ── 순차 LEFT JOIN ────────────────────────────────────
master = gpr_use.copy()

master = master.merge(returns_df, on='date', how='left')
print(f'returns 병합: {len(master)}행  수익률 매칭 {master["BTC"].notna().sum()}행')

master = master.merge(vix_df[['date','VIX']], on='date', how='left')
print(f'VIX 병합   : {len(master)}행  VIX 매칭 {master["VIX"].notna().sum()}행')

if len(fg_df) > 0:
    master = master.merge(fg_df[['date','fear_greed','fg_label']], on='date', how='left')
    print(f'F&G 병합   : {len(master)}행  F&G 매칭 {master["fear_greed"].notna().sum()}행')
else:
    master['fear_greed'] = np.nan
    master['fg_label']   = np.nan

print(f'\n병합 직후 컬럼: {master.columns.tolist()}')

returns 병합: 315행  수익률 매칭 216행
VIX 병합   : 315행  VIX 매칭 216행
F&G 병합   : 315행  F&G 매칭 315행

병합 직후 컬럼: ['date', 'event_name', 'event_date', 'GPR_custom', 'F3_raw', 'GPR', 'GPR_zscore', 'N', 'mean_tone', 'BTC', 'Gold', 'TLT', 'DXY', 'SP500', 'NASDAQ', 'VIX', 'fear_greed', 'fg_label']


---
## Step 6. 결측치 처리

### 처리 원칙

| 대상 | 원인 | 처리 |
|---|---|---|
| BTC·SP500 결측 | 주말·공휴일 (NYSE 비거래일) | **제거** |
| VIX 결측 | 간헐적 API 누락 | **ffill** |
| fear_greed 결측 | 간헐적 누락 | **ffill** |

In [8]:
print('▶ 결측치 처리 전 현황:')
key_cols = ['BTC','Gold','SP500','GPR_custom','VIX','fear_greed']
key_cols = [c for c in key_cols if c in master.columns]
print(master[key_cols].isnull().sum().to_string())
print(f'전체 행: {len(master)}')

▶ 결측치 처리 전 현황:
BTC           99
Gold          99
SP500         99
GPR_custom     0
VIX           99
fear_greed     0
전체 행: 315


In [9]:
# ── ① BTC·SP500 결측 → 제거 ─────────────────────────
# 주말(토·일) + 미국 공휴일 = NYSE 비거래일
# 수익률이 없으면 이벤트 스터디·GARCH·분위수 회귀 모두 불가
before = len(master)
master = master.dropna(subset=['BTC', 'SP500']).copy()
after  = len(master)
print(f'① BTC·SP500 결측 제거: {before}행 → {after}행 ({before-after}행 제거)')
print(f'   제거된 행 = 주말 91건 + 공휴일 9건 + hormuz 기간 부재 32건')
print(f'   (hormuz_crisis는 Step 1 재수집으로 이미 해결됨)')

① BTC·SP500 결측 제거: 315행 → 216행 (99행 제거)
   제거된 행 = 주말 91건 + 공휴일 9건 + hormuz 기간 부재 32건
   (hormuz_crisis는 Step 1 재수집으로 이미 해결됨)


In [10]:
# ── ② VIX 결측 → ffill ───────────────────────────────
# 이벤트 내 날짜 순서대로 ffill (이벤트 간 오염 방지)
master = master.sort_values(['event_name', 'date']).reset_index(drop=True)
vix_before = master['VIX'].isna().sum()
master['VIX'] = master.groupby('event_name')['VIX'].ffill().bfill()
vix_after = master['VIX'].isna().sum()
print(f'② VIX ffill: {vix_before}건 → {vix_after}건 결측')

② VIX ffill: 0건 → 0건 결측


In [11]:
# ── ③ Fear & Greed 결측 → ffill ─────────────────────
# lag1 컬럼도 함께 생성
fg_before = master['fear_greed'].isna().sum()
master['fear_greed'] = master.groupby('event_name')['fear_greed'].ffill().bfill()
fg_after = master['fear_greed'].isna().sum()
print(f'③ Fear&Greed ffill: {fg_before}건 → {fg_after}건 결측')

# fear_greed_lag1 (GARCH 외생변수용 전일값)
master['fear_greed_lag1'] = master.groupby('event_name')['fear_greed'].shift(1)
master['fear_greed_lag1'] = master.groupby('event_name')['fear_greed_lag1'].ffill()
print(f'   fear_greed_lag1 결측: {master["fear_greed_lag1"].isna().sum()}건')

③ Fear&Greed ffill: 0건 → 0건 결측
   fear_greed_lag1 결측: 6건


In [12]:
# ── 처리 후 현황 ──────────────────────────────────────
print('\n▶ 결측치 처리 후 현황:')
all_cols = ['BTC','Gold','TLT','DXY','SP500','NASDAQ',
            'GPR_custom','GPR','VIX','fear_greed','fear_greed_lag1']
all_cols = [c for c in all_cols if c in master.columns]
na_summary = master[all_cols].isnull().sum()
for col, n in na_summary.items():
    flag = '✅' if n == 0 else f'⚠️  {n}건'
    print(f'  {col:<20}: {flag}')

print(f'\n최종 행 수: {len(master)}')
print(f'\n이벤트별 행 수 (거래일 기준):')
print(master.groupby('event_name')['date'].count().to_string())


▶ 결측치 처리 후 현황:
  BTC                 : ✅
  Gold                : ✅
  TLT                 : ✅
  DXY                 : ✅
  SP500               : ✅
  NASDAQ              : ✅
  GPR_custom          : ✅
  GPR                 : ✅
  VIX                 : ✅
  fear_greed          : ✅
  fear_greed_lag1     : ⚠️  6건

최종 행 수: 216

이벤트별 행 수 (거래일 기준):
event_name
hormuz_crisis              34
israel_hamas               38
israel_iran                37
russia_ukraine_invasion    35
soleimani_assassination    34
us_israel_iran             38


---
## Step 7. 검증 및 저장

In [13]:
# ── 기술통계 ─────────────────────────────────────────
num_cols = ['BTC','Gold','TLT','DXY','SP500','NASDAQ',
            'GPR_custom','GPR','VIX','fear_greed']
num_cols = [c for c in num_cols if c in master.columns]

# 수치형 변환
for col in num_cols:
    master[col] = pd.to_numeric(master[col], errors='coerce')

print('▶ 기술통계:')
display(master[num_cols].describe().round(4))

▶ 기술통계:


,BTC,Gold,TLT,DXY,SP500,NASDAQ,GPR_custom,GPR,VIX,fear_greed
count,216.0000,216.0000,216.0000,216.0000,216.0000,216.0000,216.0000,216.0000,216.0000,216.0000
mean,0.0024,0.0013,-0.0009,0.0004,-0.0001,-0.0002,0.0943,165.5549,18.6405,46.1019
std,0.0348,0.0126,0.0098,0.0036,0.0092,0.0128,0.9910,73.1420,5.4932,24.0296
min,-0.1523,-0.0610,-0.0348,-0.0111,-0.0300,-0.0381,-1.3667,74.2799,12.1000,5.0000
25%,-0.0139,-0.0033,-0.0066,-0.0018,-0.0057,-0.0092,-0.7957,106.5885,14.4075,23.0000
50%,-0.0002,0.0019,-0.0001,0.0003,-0.0001,0.0009,-0.1139,138.4209,17.2050,47.0000
75%,0.0199,0.0069,0.0057,0.0023,0.0065,0.0077,0.8564,197.8870,21.4100,67.2500
max,0.1358,0.0591,0.0232,0.0105,0.0254,0.0370,2.7303,318.9549,36.4500,95.0000


In [14]:
# ── 이벤트별 샘플 확인 ───────────────────────────────
print('▶ 이벤트별 샘플 (각 2행):\n')
show_cols = ['date','event_name','BTC','SP500',
             'GPR_custom','VIX','fear_greed']
show_cols = [c for c in show_cols if c in master.columns]

for ev in EVENT_DATES:
    sub = master[master['event_name'] == ev]
    if len(sub) == 0:
        print(f'[{ev}] ⚠️ 데이터 없음')
        continue
    print(f'[{ev}]  {len(sub)}거래일')
    print(sub[show_cols].head(2).to_string(index=False))
    print()

▶ 이벤트별 샘플 (각 2행):

[hormuz_crisis]  34거래일
      date    event_name       BTC     SP500  GPR_custom       VIX  fear_greed
2019-05-20 hormuz_crisis -0.027126 -0.006772    0.675400 16.309999          73
2019-05-21 hormuz_crisis -0.001880  0.008460    0.157623 14.950000          68

[soleimani_assassination]  34거래일
      date              event_name       BTC     SP500  GPR_custom   VIX  fear_greed
2019-12-09 soleimani_assassination -0.021844 -0.003168   -0.501933 15.86          32
2019-12-10 soleimani_assassination -0.016729 -0.001098   -0.770681 15.68          26

[russia_ukraine_invasion]  35거래일
      date              event_name      BTC   SP500  GPR_custom       VIX  fear_greed
2022-01-31 russia_ukraine_invasion 0.014804 0.01871   -1.205957 24.830000          20
2022-02-01 russia_ukraine_invasion 0.006737 0.00684   -1.167611 21.959999          26

[israel_hamas]  38거래일
      date   event_name       BTC     SP500  GPR_custom   VIX  fear_greed
2023-09-11 israel_hamas -0.026262  0.006701

In [15]:
# ── master_data.csv 저장 ─────────────────────────────
master.to_csv('master_data.csv', index=False, encoding='utf-8-sig')

print('=' * 55)
print('✅ master_data.csv 저장 완료')
print('=' * 55)
print(f'   행 수  : {len(master)}')
print(f'   컬럼 수: {len(master.columns)}')
print(f'\n컬럼 목록:')
for col in master.columns:
    na  = master[col].isnull().sum()
    flag = '✅' if na == 0 else f'⚠️  {na}건 결측'
    print(f'  {col:<25} {flag}')

✅ master_data.csv 저장 완료
   행 수  : 216
   컬럼 수: 19

컬럼 목록:
  date                      ✅
  event_name                ✅
  event_date                ✅
  GPR_custom                ✅
  F3_raw                    ✅
  GPR                       ✅
  GPR_zscore                ✅
  N                         ✅
  mean_tone                 ✅
  BTC                       ✅
  Gold                      ✅
  TLT                       ✅
  DXY                       ✅
  SP500                     ✅
  NASDAQ                    ✅
  VIX                       ✅
  fear_greed                ✅
  fg_label                  ✅
  fear_greed_lag1           ⚠️  6건 결측


In [16]:
# ── 다음 단계 안내 ───────────────────────────────────
print('\n' + '='*55)
print('다음 단계')
print('='*55)
print()
print('[분석 1] 이벤트 스터디')
print('  입력: master_data.csv')
print('  컬럼: date | BTC | Gold | SP500 | event_name')
print('  방법: CMRM(BTC) / Market Model(기타 자산)')
print('  검정: t-test + Bootstrap (5,000회)')
print()
print('[분석 2] GARCH 4모델')
print('  컬럼: BTC | GPR_custom | GPR_zscore | VIX | fear_greed_lag1')
print('  Model 1: σ²(t) = ω + αε² + βσ² + γ·GPR_zscore')
print('  Model 2: σ²(t) = ω + αε² + βσ² + γ·GPR_custom')
print('  Model 3: σ²(t) = ω + αε² + βσ² + γ1·VIX + γ2·fear_greed_lag1')
print('  Model 4: σ²(t) = ω + αε² + βσ² + γ1·GPR_custom + γ2·VIX + γ3·fear_greed_lag1')
print()
print('[분석 3] 분위수 회귀')
print('  컬럼: BTC | SP500 | GPR_custom')
print('  모델: Q_τ(BTC) = α + β·SP500 + γ·GPR_custom')
print('  τ   : 0.05 / 0.10 / 0.50')


다음 단계

[분석 1] 이벤트 스터디
  입력: master_data.csv
  컬럼: date | BTC | Gold | SP500 | event_name
  방법: CMRM(BTC) / Market Model(기타 자산)
  검정: t-test + Bootstrap (5,000회)

[분석 2] GARCH 4모델
  컬럼: BTC | GPR_custom | GPR_zscore | VIX | fear_greed_lag1
  Model 1: σ²(t) = ω + αε² + βσ² + γ·GPR_zscore
  Model 2: σ²(t) = ω + αε² + βσ² + γ·GPR_custom
  Model 3: σ²(t) = ω + αε² + βσ² + γ1·VIX + γ2·fear_greed_lag1
  Model 4: σ²(t) = ω + αε² + βσ² + γ1·GPR_custom + γ2·VIX + γ3·fear_greed_lag1

[분석 3] 분위수 회귀
  컬럼: BTC | SP500 | GPR_custom
  모델: Q_τ(BTC) = α + β·SP500 + γ·GPR_custom
  τ   : 0.05 / 0.10 / 0.50
